Connected to pypsa-earth (Python 3.10.16)

In [1]:
# -*- coding: utf-8 -*-
# SPDX-FileCopyrightText:  PyPSA-Earth and PyPSA-Eur Authors
#
# SPDX-License-Identifier: AGPL-3.0-or-later

import os
from itertools import dropwhile
from types import SimpleNamespace

import numpy as np
import pandas as pd
import pypsa
import pytz
import xarray as xr
from _helpers import mock_snakemake, override_component_attrs


def override_values(tech, year, dr):  
    custom_res_t = pd.read_csv(
        snakemake.input["custom_res_pot_{0}_{1}_{2}".format(tech, year, dr)],
        index_col=0,
        parse_dates=True,
    )

    # custom_res_t = pd.read_csv(
    #     snakemake.input["custom_res_pot_{0}_{1}_{2}".format(tech, year, dr)],
    #     index_col=0,
    #     parse_dates=True,
    # ).filter(buses, axis=1)

    custom_res = (
        pd.read_csv(
            snakemake.input["custom_res_ins_{0}_{1}_{2}".format(tech, year, dr)],
            index_col=0,
        )
    )

    # custom_res["Generator"] = custom_res["Generator"].apply(lambda x: x + " " + tech)
    # custom_res = custom_res.set_index("Generator")

    if tech.replace("-", " ") in n.generators.carrier.unique():
        to_drop = n.generators[n.generators.carrier == tech].index
        n.mremove("Generator", to_drop)

    if snakemake.wildcards["planning_horizons"] == 2050:
        directory = "results/" + snakemake.params.run.replace("2050", "2030")
        n_name = snakemake.input.network.split("/")[-1].replace(
            n.config["scenario"]["clusters"], ""
        )
        df = pd.read_csv(directory + "/res_caps_" + n_name, index_col=0)
        # df = pd.read_csv(snakemake.config["custom_data"]["existing_renewables"], index_col=0)
        existing_res = df.loc[tech]
        existing_res.index = existing_res.index.str.apply(lambda x: x + tech)
    else:
        existing_res = custom_res["installedcapacity"].values

    n.madd(
        "Generator",
        buses,
        " " + tech,
        bus=buses,
        carrier=tech,
        p_nom_extendable=True,
        p_nom_max=custom_res["p_nom_max"].values,
        # weight=ds["weight"].to_pandas(),
        # marginal_cost=custom_res["fixedomEuroPKW"].values * 1000,
        capital_cost=custom_res["annualcostEuroPMW"].values,
        efficiency=1.0,
        p_max_pu=custom_res_t,
        lifetime=custom_res["lifetime"][0],
        p_nom_min=existing_res,
    )


if __name__ == "__main__":
    if "snakemake" not in globals():
        snakemake = mock_snakemake(
            "override_respot",
            simpl="",
            clusters="3flex",
            ll="copt",
            opts="Co2L-4H",
            planning_horizons="2030",
            sopts="144h",
            discountrate=0.071,
            demand="AB",
        )

    overrides = override_component_attrs(snakemake.input.overrides)
    n = pypsa.Network(snakemake.input.network, override_component_attrs=overrides)
    m = n.copy()
    if snakemake.params.custom_data["renewables"]:
        buses = list(n.buses[n.buses.carrier == "AC"].index)
        energy_totals = pd.read_csv(snakemake.input.energy_totals, index_col=0)
        countries = snakemake.params.countries
        if snakemake.params.custom_data["renewables"]:
            techs = snakemake.params.custom_data["renewables"]
            year = snakemake.wildcards["planning_horizons"]
            dr = snakemake.wildcards["discountrate"]

            m = n.copy()

            for tech in techs:
                override_values(tech, year, dr)

        else:
            print("No RES potential techs to override...")

        if snakemake.params.custom_data["elec_demand"]:
            for country in countries:
                n.loads_t.p_set.filter(like=country)[buses] = (
                    (
                        n.loads_t.p_set.filter(like=country)[buses]
                        / n.loads_t.p_set.filter(like=country)[buses].sum().sum()
                    )
                    * energy_totals.loc[country, "electricity residential"]
                    * 1e6
                )

    n.export_to_netcdf(snakemake.output[0])

Demand data folder: data/ssp2-2.6/2030/era5_2013, load path is ['data/ssp2-2.6/2030/era5_2013/Africa.nc'].
Expected files: Africa.nc


INFO:pypsa.io:Imported network elec_s_3flex_ec_lcopt_Co2L-4H.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores
<ipython-input-1-d7d582ce00a9>:70: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lifetime=custom_res["lifetime"][0],
               '2013-01-01 08:00:00', '2013-01-01 12:00:00',
               '2013-01-01 16:00:00', '2013-01-01 20:00:00',
               '2013-01-02 00:00:00', '2013-01-02 04:00:00',
               '2013-01-02 08:00:00', '2013-01-02 12:00:00',
               ...
               '2013-12-30 08:00:00', '2013-12-30 12:00:00',
               '2013-12-30 16:00:00', '2013-12-30 20:00:00',
               '2013-12-31 00:00:00', '2013-12-31 04:00:00',
               '2013-12-31 08:00:00', '2013-12-31 12:00:00',
               '2013-12-31 1

In [2]:
n

PyPSA Network
Components:
 - Bus: 9
 - Carrier: 17
 - Generator: 21
 - GlobalConstraint: 1
 - Line: 2
 - Link: 12
 - Load: 3
 - StorageUnit: 3
 - Store: 6
Snapshots: 2190

In [3]:
n.buses

,v_nom,tag_substation,tag_area,lon,lat,country,x,y,control,generator,carrier,location,sub_network,type,unit,v_mag_pu_max,v_mag_pu_min,v_mag_pu_set
Bus,,,,,,,,,,,,,,,,,,
MA0_AC,380.0,transmission,0.0,-10.487250,28.780483,MA,-10.487250,28.780483,Slack,MA0_AC CCGT,AC,,,,MWh,inf,0.0,1.0
MA1_AC,380.0,transmission,0.0,-7.705631,32.214609,MA,-7.705631,32.214609,PQ,,AC,,,,MWh,inf,0.0,1.0
MA2_AC,380.0,transmission,0.0,-4.795889,34.241255,MA,-4.795889,34.241255,PQ,,AC,,,,MWh,inf,0.0,1.0
MA0_AC H2,1.0,,NaN,NaN,NaN,MA,-10.487250,28.780483,PQ,,H2,,,,MWh,inf,0.0,1.0
MA1_AC H2,1.0,,NaN,NaN,NaN,MA,-7.705631,32.214609,PQ,,H2,,,,MWh,inf,0.0,1.0
MA2_AC H2,1.0,,NaN,NaN,NaN,MA,-4.795889,34.241255,PQ,,H2,,,,MWh,inf,0.0,1.0
MA0_AC battery,1.0,,NaN,NaN,NaN,MA,-10.487250,28.780483,PQ,,battery,,,,MWh,inf,0.0,1.0
MA1_AC battery,1.0,,NaN,NaN,NaN,MA,-7.705631,32.214609,PQ,,battery,,,,MWh,inf,0.0,1.0
MA2_AC battery,1.0,,NaN,NaN,NaN,MA,-4.795889,34.241255,PQ,,battery,,,,MWh,inf,0.0,1.0


In [4]:
n.carriers

,co2_emissions,color,nice_name,max_growth,max_relative_growth
Carrier,,,,,
OCGT,0.1980,#d35050,Open-Cycle Gas,inf,0.0
oil,0.2571,#262626,Oil,inf,0.0
lignite,0.4069,#9e5a01,Lignite,inf,0.0
nuclear,0.0000,#ff9000,Nuclear,inf,0.0
CCGT,0.1980,#b80404,Combined-Cycle Gas,inf,0.0
geothermal,0.1200,#ba91b1,Geothermal,inf,0.0
coal,0.3361,#707070,Coal,inf,0.0
biomass,0.0000,#0c6013,Biomass,inf,0.0
hydro,0.0000,#08ad97,Reservoir & Dam,inf,0.0


In [5]:
n.generators_t

{'efficiency': Empty DataFrame
 Columns: []
 Index: [2013-01-01 00:00:00, 2013-01-01 04:00:00, 2013-01-01 08:00:00, 2013-01-01 12:00:00, 2013-01-01 16:00:00, 2013-01-01 20:00:00, 2013-01-02 00:00:00, 2013-01-02 04:00:00, 2013-01-02 08:00:00, 2013-01-02 12:00:00, 2013-01-02 16:00:00, 2013-01-02 20:00:00, 2013-01-03 00:00:00, 2013-01-03 04:00:00, 2013-01-03 08:00:00, 2013-01-03 12:00:00, 2013-01-03 16:00:00, 2013-01-03 20:00:00, 2013-01-04 00:00:00, 2013-01-04 04:00:00, 2013-01-04 08:00:00, 2013-01-04 12:00:00, 2013-01-04 16:00:00, 2013-01-04 20:00:00, 2013-01-05 00:00:00, 2013-01-05 04:00:00, 2013-01-05 08:00:00, 2013-01-05 12:00:00, 2013-01-05 16:00:00, 2013-01-05 20:00:00, 2013-01-06 00:00:00, 2013-01-06 04:00:00, 2013-01-06 08:00:00, 2013-01-06 12:00:00, 2013-01-06 16:00:00, 2013-01-06 20:00:00, 2013-01-07 00:00:00, 2013-01-07 04:00:00, 2013-01-07 08:00:00, 2013-01-07 12:00:00, 2013-01-07 16:00:00, 2013-01-07 20:00:00, 2013-01-08 00:00:00, 2013-01-08 04:00:00, 2013-01-08 08:00:00, 20

In [6]:
n.generators_t.p_max_pu

Generator,MA0_AC offwind-ac,MA0_AC offwind-dc,MA1_AC offwind-ac,MA1_AC offwind-dc,MA2_AC offwind-ac,MA2_AC offwind-dc,MAR0_AC onwind onwind,MAR0_AC solar solar,MAR0_offgrid onwind onwind,MAR1_AC onwind onwind,MAR1_AC solar solar,MAR1_offgrid onwind onwind,MAR2_AC onwind onwind,MAR2_AC solar solar,MAR2_offgrid onwind onwind
snapshot,,,,,,,,,,,,,,,
2013-01-01 00:00:00,0.0,0.496649,0.0,0.236766,0.0,0.055555,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2013-01-01 04:00:00,0.0,0.561091,0.0,0.285235,0.0,0.282068,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2013-01-01 08:00:00,0.0,0.696255,0.0,0.260060,0.0,0.172649,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2013-01-01 12:00:00,0.0,0.812813,0.0,0.315456,0.0,0.280880,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2013-01-01 16:00:00,0.0,0.881071,0.0,0.440429,0.0,0.286081,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-31 04:00:00,0.0,0.000312,0.0,0.020313,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2013-12-31 08:00:00,0.0,0.018013,0.0,0.006226,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2013-12-31 12:00:00,0.0,0.066186,0.0,0.012075,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [7]:
n.generators_t.p_max_pu.columns

Index(['MA0_AC offwind-ac', 'MA0_AC offwind-dc', 'MA1_AC offwind-ac',
       'MA1_AC offwind-dc', 'MA2_AC offwind-ac', 'MA2_AC offwind-dc',
       'MAR0_AC onwind onwind', 'MAR0_AC solar solar',
       'MAR0_offgrid onwind onwind', 'MAR1_AC onwind onwind',
       'MAR1_AC solar solar', 'MAR1_offgrid onwind onwind',
       'MAR2_AC onwind onwind', 'MAR2_AC solar solar',
       'MAR2_offgrid onwind onwind'],
      dtype='object', name='Generator')

In [8]:
# -*- coding: utf-8 -*-
# SPDX-FileCopyrightText:  PyPSA-Earth and PyPSA-Eur Authors
#
# SPDX-License-Identifier: AGPL-3.0-or-later

import os
from itertools import dropwhile
from types import SimpleNamespace

import numpy as np
import pandas as pd
import pypsa
import pytz
import xarray as xr
from _helpers import mock_snakemake, override_component_attrs


def override_values(tech, year, dr):  
    custom_res_t = pd.read_csv(
        snakemake.input["custom_res_pot_{0}_{1}_{2}".format(tech, year, dr)],
        index_col=0,
        parse_dates=True,
    )

    # custom_res_t = pd.read_csv(
    #     snakemake.input["custom_res_pot_{0}_{1}_{2}".format(tech, year, dr)],
    #     index_col=0,
    #     parse_dates=True,
    # ).filter(buses, axis=1)

    custom_res = (
        pd.read_csv(
            snakemake.input["custom_res_ins_{0}_{1}_{2}".format(tech, year, dr)],
            index_col=0,
        )
    )

    # custom_res["Generator"] = custom_res["Generator"].apply(lambda x: x + " " + tech)
    # custom_res = custom_res.set_index("Generator")

    if tech.replace("-", " ") in n.generators.carrier.unique():
        to_drop = n.generators[n.generators.carrier == tech].index
        n.mremove("Generator", to_drop)

    if snakemake.wildcards["planning_horizons"] == 2050:
        directory = "results/" + snakemake.params.run.replace("2050", "2030")
        n_name = snakemake.input.network.split("/")[-1].replace(
            n.config["scenario"]["clusters"], ""
        )
        df = pd.read_csv(directory + "/res_caps_" + n_name, index_col=0)
        # df = pd.read_csv(snakemake.config["custom_data"]["existing_renewables"], index_col=0)
        existing_res = df.loc[tech]
        existing_res.index = existing_res.index.str.apply(lambda x: x + tech)
    else:
        existing_res = custom_res["installedcapacity"].values

    n.madd(
        "Generator",
        buses,
        " " + tech,
        bus=buses,
        carrier=tech,
        p_nom_extendable=True,
        p_nom_max=custom_res["p_nom_max"].values,
        # weight=ds["weight"].to_pandas(),
        # marginal_cost=custom_res["fixedomEuroPKW"].values * 1000,
        capital_cost=custom_res["annualcostEuroPMW"].values,
        efficiency=1.0,
        p_max_pu=custom_res_t,
        lifetime=custom_res["lifetime"][0],
        p_nom_min=existing_res,
    )


if __name__ == "__main__":
    if "snakemake" not in globals():
        snakemake = mock_snakemake(
            "override_respot",
            simpl="",
            clusters="3flex",
            ll="copt",
            opts="Co2L-4H",
            planning_horizons="2030",
            sopts="144h",
            discountrate=0.071,
            demand="AB",
        )

    overrides = override_component_attrs(snakemake.input.overrides)
    n = pypsa.Network(snakemake.input.network, override_component_attrs=overrides)
    m = n.copy()
    if snakemake.params.custom_data["renewables"]:
        buses = list(n.buses[n.buses.carrier == "AC"].index)
        energy_totals = pd.read_csv(snakemake.input.energy_totals, index_col=0)
        countries = snakemake.params.countries
        if snakemake.params.custom_data["renewables"]:
            techs = snakemake.params.custom_data["renewables"]
            year = snakemake.wildcards["planning_horizons"]
            dr = snakemake.wildcards["discountrate"]

            m = n.copy()

            for tech in techs:
                override_values(tech, year, dr)

        else:
            print("No RES potential techs to override...")

        if snakemake.params.custom_data["elec_demand"]:
            for country in countries:
                n.loads_t.p_set.filter(like=country)[buses] = (
                    (
                        n.loads_t.p_set.filter(like=country)[buses]
                        / n.loads_t.p_set.filter(like=country)[buses].sum().sum()
                    )
                    * energy_totals.loc[country, "electricity residential"]
                    * 1e6
                )

    n.export_to_netcdf(snakemake.output[0])

INFO:pypsa.io:Imported network elec_s_3flex_ec_lcopt_Co2L-4H.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores
<ipython-input-8-d7d582ce00a9>:70: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lifetime=custom_res["lifetime"][0],
               '2013-01-01 08:00:00', '2013-01-01 12:00:00',
               '2013-01-01 16:00:00', '2013-01-01 20:00:00',
               '2013-01-02 00:00:00', '2013-01-02 04:00:00',
               '2013-01-02 08:00:00', '2013-01-02 12:00:00',
               ...
               '2013-12-30 08:00:00', '2013-12-30 12:00:00',
               '2013-12-30 16:00:00', '2013-12-30 20:00:00',
               '2013-12-31 00:00:00', '2013-12-31 04:00:00',
               '2013-12-31 08:00:00', '2013-12-31 12:00:00',
               '2013-12-31 1

In [9]:
n.generators_t.p_max_pu.columns

Index(['MA0_AC offwind-ac', 'MA0_AC offwind-dc', 'MA1_AC offwind-ac',
       'MA1_AC offwind-dc', 'MA2_AC offwind-ac', 'MA2_AC offwind-dc',
       'MAR0_AC onwind onwind', 'MAR0_AC solar solar', 'MAR1_AC onwind onwind',
       'MAR1_AC solar solar', 'MAR2_AC onwind onwind', 'MAR2_AC solar solar'],
      dtype='object', name='Generator')

In [10]:
n.generators_t.p_max_pu

Generator,MA0_AC offwind-ac,MA0_AC offwind-dc,MA1_AC offwind-ac,MA1_AC offwind-dc,MA2_AC offwind-ac,MA2_AC offwind-dc,MAR0_AC onwind onwind,MAR0_AC solar solar,MAR1_AC onwind onwind,MAR1_AC solar solar,MAR2_AC onwind onwind,MAR2_AC solar solar
snapshot,,,,,,,,,,,,
2013-01-01 00:00:00,0.0,0.496649,0.0,0.236766,0.0,0.055555,1.0,1.0,1.0,1.0,1.0,1.0
2013-01-01 04:00:00,0.0,0.561091,0.0,0.285235,0.0,0.282068,1.0,1.0,1.0,1.0,1.0,1.0
2013-01-01 08:00:00,0.0,0.696255,0.0,0.260060,0.0,0.172649,1.0,1.0,1.0,1.0,1.0,1.0
2013-01-01 12:00:00,0.0,0.812813,0.0,0.315456,0.0,0.280880,1.0,1.0,1.0,1.0,1.0,1.0
2013-01-01 16:00:00,0.0,0.881071,0.0,0.440429,0.0,0.286081,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-31 04:00:00,0.0,0.000312,0.0,0.020313,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0
2013-12-31 08:00:00,0.0,0.018013,0.0,0.006226,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0
2013-12-31 12:00:00,0.0,0.066186,0.0,0.012075,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0
